Red convolucional

In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import kagglehub
import os
import pandas as pd
from PIL import Image
import torch.nn.functional as F

In [5]:
# --- Configuración del Dataset de KaggleHub ---
# Reemplaza con el ID de tu dataset en KaggleHub. 
# Formato: 'owner/dataset-slug/version' o 'owner/dataset-slug'
KAGGLE_DATASET_ID = 'pkdarabi/cardetection' 

# 1. Descargar el dataset usando kagglehub
# Esto descargará y descomprimirá el dataset en una ubicación temporal/cache.
print(f"Descargando dataset: {KAGGLE_DATASET_ID}")
# 'download' devuelve la ruta local donde se guardó el dataset.
KAGGLE_DOWNLOAD_PATH = kagglehub.dataset_download(KAGGLE_DATASET_ID)
print(f"Dataset descargado en: {KAGGLE_DOWNLOAD_PATH}")


# --- CÓDIGO PARA GENERAR EL DATASET ---
IMAGE_SIZE = (416, 416) 
DATA_DIR = './'
LABELS_NAME = 'yolo_labels_dataset.csv'

# 1. Definir transformaciones
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(), 
])


# 2. Clase para Detección de Objetos (SIN CAMBIOS)
class YOLODataset(Dataset):
    def __init__(self, archivo_csv, directorio_imagenes, transform=None):
        self.full_labels_df = pd.read_csv(archivo_csv)
        self.directorio_imagenes = directorio_imagenes
        self.transform = transform

        self.imagenes_unicas = self.full_labels_df['nombre_archivo'].unique()
        self.labels_grouped = self.full_labels_df.groupby('nombre_archivo')

    def __len__(self):
        return len(self.imagenes_unicas)

    def __getitem__(self, idx):
        image_name = self.imagenes_unicas[idx]
        image_path = os.path.join(self.directorio_imagenes, image_name)
        image = Image.open(image_path).convert('RGB')
        
        boxes_df = self.labels_grouped.get_group(image_name)
        
        clase_idx = int(boxes_df['clase_indice'].iloc[0])
        label = torch.tensor(clase_idx, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label


# 2. Definir las rutas usando la ruta de descarga de Kaggle
ruta_csv = os.path.join(DATA_DIR, LABELS_NAME) # El CSV se creó en el directorio actual
# IMPORTANTE: Definir la ruta de imágenes APUNTANDO al subdirectorio 'train/images'
ruta_imgs = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'train', 'images') 

# 3. Crear instancia del Dataset
if not os.path.exists(ruta_csv) or not os.path.exists(ruta_imgs):
    print("Error: Asegúrate de que el CSV existe y que la ruta de imágenes de Kaggle es correcta.")
else:
    dataset_yolo = YOLODataset(archivo_csv=ruta_csv, directorio_imagenes=ruta_imgs, transform=transform)
    print(f"\nTipo de objeto creado: {type(dataset_yolo)}")
    print(f"Número total de imágenes (longitud del dataset): {len(dataset_yolo)}")


Descargando dataset: pkdarabi/cardetection
Dataset descargado en: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5

Tipo de objeto creado: <class '__main__.YOLODataset'>
Número total de imágenes (longitud del dataset): 3527


In [6]:
# --- Configuración del Loader ---
BATCH_SIZE = 16 # Tamaño de batch comúnmente usado en detección de objetos.

# Crear el DataLoader
train_loader = DataLoader(dataset_yolo, batch_size=BATCH_SIZE, shuffle=True)

print(f"\nDataLoader creado con éxito. Número de batches: {len(train_loader)}")


DataLoader creado con éxito. Número de batches: 221


In [7]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        # Primera capa convolutiva
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # Max pooling de 2x2
        # Segunda capa convolutiva
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Capa completamente conectada
        self.fc1 = nn.Linear(64 * 104 * 104, 120)  # Ajustamos correctamente las dimensiones
        self.fc2 = nn.Linear(120, 15)  # 15 categorías de salida

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # Conv1 -> ReLU -> Pooling
        x = self.pool(F.relu(self.conv2(x)))  # Conv2 -> ReLU -> Pooling
        x = x.view(-1, 64 * 104 * 104)  # Aplanamos el tensor
        x = F.relu(self.fc1(x))  # FC1 -> ReLU
        x = self.fc2(x)  # FC2
        return x

In [8]:
# Instanciamos la red y configuramos el entrenamiento
model = ConvNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Entrenamiento
epochs = 40
for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:  # Asegúrate de que `train_loader` esté correctamente definido
        optimizer.zero_grad()  # Limpiamos los gradientes
        outputs = model(images)  # Pasamos las imágenes por la red
        loss = criterion(outputs, labels)  # Calculamos la pérdida
        loss.backward()  # Backpropagation
        optimizer.step()  # Actualizamos los pesos

        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss / len(train_loader)}')

Epoch 1, Loss: 2.476525477154762
Epoch 2, Loss: 1.6958120664859788
Epoch 3, Loss: 1.199205797452193
Epoch 4, Loss: 0.6972223429927998
Epoch 5, Loss: 0.3850776702249886
Epoch 6, Loss: 0.25472473877806606
Epoch 7, Loss: 0.1893362277967021
Epoch 8, Loss: 0.15575871300752953
Epoch 9, Loss: 0.1024858111281757
Epoch 10, Loss: 0.1063805805657236
Epoch 11, Loss: 0.09008203545133309
Epoch 12, Loss: 0.09436877746012787
Epoch 13, Loss: 0.11938132171474852
Epoch 14, Loss: 0.1407146173844333
Epoch 15, Loss: 0.06329090129273282
Epoch 16, Loss: 0.05410936498912831
Epoch 17, Loss: 0.060985499384815116
Epoch 18, Loss: 0.039992840466515175
Epoch 19, Loss: 0.04479353127358308
Epoch 20, Loss: 0.04404553123906506
Epoch 21, Loss: 0.05108949251797438
Epoch 22, Loss: 0.0967979999920535
Epoch 23, Loss: 0.1412093515525787
Epoch 24, Loss: 0.07469683110622301
Epoch 25, Loss: 0.06790913786402376
Epoch 26, Loss: 0.029649551007263224
Epoch 27, Loss: 0.03618424798806943
Epoch 28, Loss: 0.028901871803462372
Epoch 29, 

In [11]:
# --- Configuración de rutas para test ---
IMAGEN_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 
ETIQUETAS_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'labels')
CSV_SALIDA_TEST = 'yolo_labels_test.csv'

# Definir las rutas usando la ruta de descarga de Kaggle
ruta_csv_test = os.path.join(DATA_DIR, CSV_SALIDA_TEST) # El CSV se creó en el directorio actual
# IMPORTANTE: Definir la ruta de imágenes APUNTANDO al subdirectorio 'train/images'
ruta_imgs_test = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 

# Crear DataLoader de test
dataset_yolo_test = YOLODataset(archivo_csv=ruta_csv_test, directorio_imagenes=ruta_imgs_test, transform=transform)
test_loader = DataLoader(dataset_yolo_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTest DataLoader creado con {len(test_loader)} batches")



Test DataLoader creado con 40 batches


In [14]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Evaluación de la red
def evaluate(model, test_loader):
    model.eval()  # Poner el modelo en modo evaluación
    correct = 0
    total = test_loader.dataset.__len__()  # Total de muestras en el conjunto de test
    print(f'Total de muestras en el conjunto de test: {total}')
    with torch.no_grad():  # No calcular gradientes
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)  # Mover datos al dispositivo
            outputs = model(inputs)  # Forward pass
            _, predicted = torch.max(outputs.data, 1)  # Obtener las predicciones
            correct += (predicted == labels).sum().item()  # Actualizar el contador de aciertos
    accuracy = 100 * correct / total if total > 0 else 0
    print(f'Accuracy: {accuracy:.2f}%')

In [15]:
evaluate(model, test_loader)

Total de muestras en el conjunto de test: 637
Accuracy: 51.65%
